In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             classification_report, confusion_matrix)

RANDOM_STATE = 42


data = load_breast_cancer(as_frame=True)
df = data.frame
TARGET_COL = "target" 

print("STEP 1 | Patients:", df.shape[0], "| Features:", df.shape[1] - 1)
print(df.iloc[:, :5].head(), "\n")
print("Missing values in dataset:", int(df.isna().sum().sum()))
print("Class balance:\n", df[TARGET_COL].value_counts().rename(
      {0: "Disease (malignant)", 1: "Healthy (benign)"}), "\n")



X = df.drop(columns=TARGET_COL)
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print(f"STEP 2 | Train: {len(X_train)} patients | Test: {len(X_test)} patients\n")



models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                    probability=True, random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
}

try: 
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=3,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=RANDOM_STATE)
except ImportError:
    print("NOTE: xgboost not installed — skipping it (`pip install xgboost` to add).\n")


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results, roc_data = [], {}

for name, model in models.items():
    cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model":      name,
        "CV ROC-AUC": cv_auc.mean(),
        "Accuracy":   accuracy_score(y_test, y_pred),
        "Precision":  precision_score(y_test, y_pred),
        "Recall":     recall_score(y_test, y_pred),
        "F1-Score":   f1_score(y_test, y_pred),
        "ROC-AUC":    roc_auc_score(y_test, y_proba),
    })
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, roc_auc_score(y_test, y_proba))
    print(f"{name:22s} | CV AUC {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})"
          f" | Test AUC {roc_auc_score(y_test, y_proba):.4f}")

scores = pd.DataFrame(results).set_index("Model").round(4)
print("\nSTEP 4 | MODEL COMPARISON\n", scores, "\n")

best_name  = scores["ROC-AUC"].idxmax()
best_model = models[best_name]
print(f"BEST MODEL: {best_name}\n")
print("Detailed report for the best model:")
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=["Disease (0)", "Healthy (1)"], digits=4))


fig, axes = plt.subplots(1, 3, figsize=(19, 5))

cm = confusion_matrix(y_test, best_model.predict(X_test))
axes[0].imshow(cm, cmap="Blues")
axes[0].set_title(f"Confusion Matrix — {best_name}")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1], ["Disease", "Healthy"])
axes[0].set_yticks([0, 1], ["Disease", "Healthy"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="red", fontsize=16, fontweight="bold")

for name, (fpr, tpr, auc) in roc_data.items():
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC Curves"); axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate"); axes[1].legend(loc="lower right", fontsize=8)

rf = models["Random Forest"]
top = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10).sort_values()
axes[2].barh(top.index, top.values, color="darkorange")
axes[2].set_title("Top 10 Important Medical Features")

plt.tight_layout()
plt.savefig("task4_disease_prediction_results.png", dpi=120)
print("STEP 5 | Plots saved -> task4_disease_prediction_results.png\n")


new_patient = X_test.iloc[[0]]                     
pred  = best_model.predict(new_patient)[0]
proba = best_model.predict_proba(new_patient)[0]

print("STEP 6 | NEW PATIENT PREDICTION")
print(f"  Result      : {'HEALTHY (benign)' if pred == 1 else 'DISEASE DETECTED (malignant)'}")
print(f"  Probability : disease {proba[0]:.2%} | healthy {proba[1]:.2%}")
print("  (Screening support only — not a medical diagnosis.)")

STEP 1 | Patients: 569 | Features: 30
   mean radius  mean texture  mean perimeter  mean area  mean smoothness
0        17.99         10.38          122.80     1001.0          0.11840
1        20.57         17.77          132.90     1326.0          0.08474
2        19.69         21.25          130.00     1203.0          0.10960
3        11.42         20.38           77.58      386.1          0.14250
4        20.29         14.34          135.10     1297.0          0.10030 

Missing values in dataset: 0
Class balance:
 target
Healthy (benign)       357
Disease (malignant)    212
Name: count, dtype: int64 

STEP 2 | Train: 455 patients | Test: 114 patients

Logistic Regression    | CV AUC 0.9959 (+/- 0.0050) | Test AUC 0.9954
SVM (RBF)              | CV AUC 0.9956 (+/- 0.0048) | Test AUC 0.9950
Random Forest          | CV AUC 0.9906 (+/- 0.0064) | Test AUC 0.9934
XGBoost                | CV AUC 0.9941 (+/- 0.0054) | Test AUC 0.9940

STEP 4 | MODEL COMPARISON
                      CV ROC-A